# Day 28 — Hypothesis Testing
### Statistics & Probability · Module 2 · Topic 2.8

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | The logic and the formula | 15 min |
| 2 | **What a p-value is** — and the four things it is not | 20 min |
| 3 | Choosing the right test — Welch, paired, t vs z | 20 min |
| 4 | The two errors, and power | 15 min |
| 5 | ⚠️ **How it goes wrong** — multiplicity, p-hacking, base rates | 20 min |
| 6 | **Practice notebook (separate file)** | 30 min |

---

## The step we take today

Yesterday you learned how far a sample mean typically lands from the truth: **σ/√n**. Today
that becomes a decision procedure.

The whole formula is:

$$t \;=\; \frac{\text{what we saw} \;-\; \text{what we would expect}}{\text{SE}}$$

*"How many standard errors away from boring is this?"* You already have every piece.

### So why does this need two hours?

Because this is the single most misused tool in applied statistics. Entire research
literatures have been built on p-values that did not mean what their authors believed. The
formula takes ten minutes; knowing when it lies to you takes the rest of the session.

### Four claims to hold me to

1. A p-value is **not** the probability that your hypothesis is wrong.
2. p = 0.049 and p = 0.051 differ by **0.017** in the test statistic.
3. Two completely ordinary research habits **triple** your false positive rate.
4. In some fields, **most** "significant" findings are false — and that is arithmetic, not
   an accusation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

plt.rcParams["figure.figsize"] = (8, 3.2)
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# 1. The logic

Assume nothing is happening. Work out what the data would look like under that assumption. Then
see whether what you actually got is a comfortable fit or an awkward one.

It is a proof by contradiction, run on probabilities instead of logic.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3))
xs = np.linspace(-4.2, 4.2, 600)
pdf = stats.norm.pdf(xs)
obs = 2.4
ax.fill_between(xs, 0, pdf, color="#D8E1EB")
ax.fill_between(xs[xs >= obs], 0, pdf[xs >= obs], color="#E8674C")
ax.fill_between(xs[xs <= -obs], 0, pdf[xs <= -obs], color="#E8674C")
ax.plot(xs, pdf, color="#1F3A5F", lw=2)
ax.axvline(obs, color="#152238", lw=2.2)
ax.text(0, 0.14, "what we would expect\nif nothing were happening",
        ha="center", fontsize=11, color="#1F3A5F")
ax.set_yticks([]); ax.set_xlabel("test statistic")
ax.set_title("The p-value is the red area: how often nothing looks like this")
plt.show()

print(f"p-value for a statistic of {obs}: {2*(1-stats.norm.cdf(obs)):.4f}")

## The four steps

| # | Step | What it means |
|---|---|---|
| 1 | **State the null** | The boring version of the world: no difference, no effect. |
| 2 | **Compute the statistic** | The gap, divided by yesterday's SE. |
| 3 | **Find the p-value** | How often a world with no effect produces a gap this big. |
| 4 | **Decide — and report more** | The decision is the *least* interesting output. |

## A worked test: did the training course help?

60 employees, productivity scored before and after a course. Each person is measured twice, so
the test is **paired**.

In [ ]:
rng = np.random.default_rng(2801)
before = rng.normal(72, 11, 60)
after  = before + rng.normal(3.2, 6, 60)      # a real effect, but a noisy one

change = after - before

#  --- 1. the null: the course does nothing, so the mean change is 0
#  --- 2. the statistic
mean_d = change.mean()
sd_d   = change.std(ddof=1)
se     = sd_d / np.sqrt(60)
t_hand = mean_d / se

print(f"mean change  {mean_d:+.3f}")
print(f"sd of change  {sd_d:.3f}")
print(f"standard err  {se:.3f}")
print(f"t (by hand)  {t_hand:+.3f}")

In [ ]:
#  --- 3. the p-value
res = stats.ttest_rel(after, before)
print(f"t = {res.statistic:.3f}   df = {59}   p = {res.pvalue:.5f}")
print()

#  --- 4. the part that actually matters
lo, hi = stats.t.interval(0.95, 59, mean_d, se)
cohen = mean_d / sd_d
print(f"95% CI for the change: {lo:+.2f} to {hi:+.2f}")
print(f"Cohen's d:             {cohen:.3f}")
print()
print(f"In words: the course raised scores by about {mean_d:.1f} points,")
print(f"and we are confident the true gain is between {lo:.0f} and {hi:.0f}.")

Notice that the final sentence contains **no p-value at all**, and is more informative than one.
That is the habit this session is trying to build.

---
# 2. What a p-value is

The cleanest definition is a fact most people have never seen: **under the null, p-values are
uniformly distributed.** Every value between 0 and 1 is equally likely.

That is Day 26's flat line, and it is the definition working.

In [ ]:
rng = np.random.default_rng(28)

fig, axes = plt.subplots(1, 4, figsize=(13, 2.7), sharey=True)
for ax, d in zip(axes, [0.0, 0.2, 0.5, 0.8]):
    a = rng.normal(0, 1, (30_000, 30))
    b = rng.normal(d, 1, (30_000, 30))
    p = stats.ttest_ind(a, b, axis=1).pvalue
    ax.hist(p, bins=20, range=(0, 1), color="#9AA7B8" if d == 0 else "#3E7CB1")
    ax.axhline(30_000/20, color="#152238", ls="--", lw=1.5)
    ax.axvline(0.05, color="#E8674C", lw=2)
    ax.set_title(f"effect = {d}\nP(p<0.05) = {(p<0.05).mean():.2f}", fontsize=10)
    ax.set_xlabel("p-value")
plt.tight_layout(); plt.show()

In [ ]:
#  check the uniformity precisely
a = rng.normal(0, 1, (40_000, 30)); b = rng.normal(0, 1, (40_000, 30))
p = stats.ttest_ind(a, b, axis=1).pvalue
print(f"KS test against uniform: p = {stats.kstest(p, 'uniform').pvalue:.3f}  (large = uniform)")
print()
for alpha in [0.01, 0.05, 0.10, 0.50]:
    print(f"  P(p < {alpha:<4}) = {(p<alpha).mean():.4f}   should be {alpha}")
print()
print("5% of the time, a world with nothing in it hands you p < 0.05.")
print("That is not a flaw. That is the definition.")

## ⚠️ Four things a p-value is **not**

| ❌ The misreading | Why it is wrong |
|---|---|
| "p = 0.03, so there is a 3% chance the null is true" | Backwards. p **assumes** the null and asks how odd the data is. It cannot judge the null. |
| "p = 0.03, so there is a 97% chance the effect is real" | That depends on how plausible the effect was beforehand. Section 5 puts a number on it. |
| "p = 0.20, so there is no effect" | You did not *find* one, which may only mean your study was small. |
| "p = 0.001, so the effect is large" | With enough data a meaningless difference gets a tiny p. |

✅ **What it IS:** *if nothing were happening, how often would I see data at least this
extreme?* That sentence, and nothing more.

It is long and awkward, and that awkwardness is exactly why people substitute the shorter,
wrong versions.

Formally: **p = P(data | null)**, not **P(null | data)**. Those are different quantities, and
section 5 shows just how different.

## ⚠️ There is nothing magic about 0.05

In [ ]:
z1 = stats.norm.ppf(1 - 0.049/2)
z2 = stats.norm.ppf(1 - 0.051/2)
print(f"a result with p = 0.049 has a test statistic of {z1:.4f}")
print(f"a result with p = 0.051 has a test statistic of {z2:.4f}")
print(f"the difference between 'a discovery' and 'nothing': {z1-z2:.4f}")
print()
print("Fisher proposed 0.05 in 1925 as a convenient rule of thumb.")
print("Particle physics uses 0.0000003. The threshold should follow")
print("from the cost of being wrong, not from habit.")

**The practical habit:** report the actual p-value, not "p < 0.05". A reader can then judge
0.049 and 0.004 differently — which they should, because they are different.

---
# 3. Choosing the test

| Your question | The test | scipy |
|---|---|---|
| Is this mean different from a target? | one-sample t | `ttest_1samp(x, mu0)` |
| Are these two groups different? | **Welch's t** | `ttest_ind(a, b, equal_var=False)` |
| Same people, before and after? | **paired t** | `ttest_rel(after, before)` |
| Are these two proportions different? | two-proportion z | `proportions_ztest` |
| More than two groups? | ANOVA, then adjust | `f_oneway` |
| Counts in categories? | chi-square | `chi2_contingency` |
| Badly non-normal, small n? | permutation test | `permutation_test` |

The next three subsections are the choices that most often get made wrong — and in each case
the wrong choice runs without error and returns a confident number.

## ⚠️ 3.1 scipy's default two-sample test is the wrong one

In [ ]:
import inspect
print("scipy's ttest_ind signature default:",
      inspect.signature(stats.ttest_ind).parameters['equal_var'].default)
print("-> Student's t, which assumes the two groups have equal variance.")

In [ ]:
rng = np.random.default_rng(286)
T = 30_000

print("With NO real effect at all, how often does each test fire?")
print(f"{'setup':<34} {'Student':>9} {'Welch':>8}")
for na, sa, nb, sb in [(10, 5, 50, 1), (5, 4, 45, 1), (50, 1, 50, 5), (10, 1, 50, 5)]:
    a = rng.normal(0, sa, (T, na))
    b = rng.normal(0, sb, (T, nb))
    s_ = (stats.ttest_ind(a, b, axis=1, equal_var=True).pvalue  < 0.05).mean()
    w_ = (stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue < 0.05).mean()
    flag = "  <- INFLATED" if s_ > 0.07 else ""
    print(f"n={na:>2} sd={sa}  vs  n={nb:>2} sd={sb:<12} {s_:>9.3f} {w_:>8.3f}{flag}")
print("\n(both should be 0.050)")

**When the smaller group has the larger variance, Student's t fires nine times too often.** On
data with no effect whatsoever.

The fix is one argument: `equal_var=False`. Welch holds at 5% in every case above, and costs
almost nothing when the variances genuinely do match. **Use Welch unless you have a specific
reason not to.**

## 3.2 Paired data: the difference between 0% and 98%

In [ ]:
rng = np.random.default_rng(285)
T, n = 4000, 20

#  20 people measured twice. People differ by ~20 points; the effect is +4.
person = rng.normal(100, 20, (T, n))
before = person + rng.normal(0, 3, (T, n))
after  = person + rng.normal(0, 3, (T, n)) + 4

p_unpaired = stats.ttest_ind(before, after, axis=1).pvalue
p_paired   = stats.ttest_rel(before, after, axis=1).pvalue

print(f"the effect is real and equal to +4 in every single simulation")
print()
print(f"unpaired t-test detects it {(p_unpaired<0.05).mean():>6.1%} of the time")
print(f"paired   t-test detects it {(p_paired  <0.05).mean():>6.1%} of the time")

The unpaired test has to see a 4-point signal through 20 points of between-person noise, and it
cannot. The paired test subtracts each person from themselves, the between-person variation
cancels completely, and all that is left is the change.

**If the data is paired, pairing it is free statistical power.** This is the largest single
gain available from a design choice anywhere in this module.

## 3.3 Why t and not z

In [ ]:
rng = np.random.default_rng(284)
print(f"{'n':>6} {'using 1.96':>13} {'using t':>10}")
for n in [3, 5, 10, 30, 100]:
    s = rng.normal(0, 1, (40_000, n))
    se_ = s.std(ddof=1, axis=1)/np.sqrt(n)
    z_cov = (np.abs(s.mean(axis=1)) <= 1.96*se_).mean()
    t_cov = (np.abs(s.mean(axis=1)) <= stats.t.ppf(.975, n-1)*se_).mean()
    print(f"{n:>6} {z_cov:>12.1%} {t_cov:>10.1%}")
print()
print("Both claim to be 95% intervals.")

When you use 1.96 you are pretending you know σ. You do not — you estimated it from the same
small sample, and that estimate is itself uncertain. The t distribution has heavier tails to
pay for it.

Past about n = 100 the two agree to the third decimal. Below n = 20 it is the difference
between a 95% interval and a 92% one. If you use scipy's `ttest_*` and `t.interval`, this is
already handled for you.

---
# 4. The two errors, and power

| | The effect is real | Nothing is happening |
|---|---|---|
| **Test fires** | correct | ⚠️ **Type I** — false alarm (α, the 5%) |
| **Test is silent** | ⚠️ **Type II** — a miss | correct |

You control **Type I** directly: it is α. You almost never control **Type II**, and almost
nobody reports it. That asymmetry is the problem.

**Power** = 1 − P(Type II) = the chance of noticing an effect that is really there.

In [ ]:
def power_analytic(n, d, alpha=0.05):
    crit = stats.norm.ppf(1 - alpha/2)
    ncp  = d * np.sqrt(n/2)
    return stats.norm.cdf(ncp - crit) + stats.norm.cdf(-ncp - crit)

print("Power of a two-sample test, by sample size per group:")
print(f"{'n/group':>9}" + "".join(f"{'d='+str(d):>9}" for d in [0.2, 0.5, 0.8]))
for n in [10, 20, 30, 50, 100, 200, 400]:
    print(f"{n:>9}" + "".join(f"{power_analytic(n,d):>9.2f}" for d in [0.2, 0.5, 0.8]))

print("\nn needed for the usual 80% bar:")
for d, label in [(0.8, "large"), (0.5, "medium"), (0.2, "small")]:
    n = next(n for n in range(4, 5000) if power_analytic(n, d) >= 0.80)
    print(f"  d = {d} ({label:<6}): {n:>4} per group")

Cohen's d is the gap measured in standard deviations: 0.2 small, 0.5 medium, 0.8 large.

Halve the effect and you quadruple the sample needed — the same √n law as yesterday, seen from
the other side. **Small effects are expensive to find.**

Do this calculation **before** you collect data. Afterwards it only tells you what you already
know: that you did not find anything.

## ⚠️ "Not significant" does not mean "no effect"

A null result is compatible with two completely different worlds: there is no effect, or there
is one and your study was too small to see it. **The p-value cannot tell you which. The
confidence interval can.**

In [ ]:
rng = np.random.default_rng(287)

#  two studies, same non-significant verdict, completely different information
big   = rng.normal(0.02, 1, 5000)       # large study, genuinely tiny effect
small = rng.normal(0.02, 1, 12)         # tiny study, learned nothing

for name, d in [("large study (n=5000)", big), ("tiny study  (n=12)  ", small)]:
    res = stats.ttest_1samp(d, 0)
    lo, hi = stats.t.interval(0.95, len(d)-1, d.mean(), d.std(ddof=1)/np.sqrt(len(d)))
    print(f"{name}:  p = {res.pvalue:.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")
print()
print("Same verdict: 'not significant'.")
print("First: any real effect is tiny. That is a finding.")
print("Second: the effect could be anything from strongly negative to strongly")
print("positive. That is not a finding - it is an admission.")

---
# 5. ⚠️ How it goes wrong

Three ways, and none of them feel like cheating.

## 5.1 Multiple comparisons

In [ ]:
print(f"{'tests run':>11} {'P(at least one false positive)':>32}")
for k in [1, 5, 10, 20, 50, 100]:
    print(f"{k:>11} {1-0.95**k:>31.1%}")
print()

#  and confirm it by simulation
rng = np.random.default_rng(281)
T, K, n = 5000, 20, 30
a = rng.normal(0, 1, (T, K, n)); b = rng.normal(0, 1, (T, K, n))
p = stats.ttest_ind(a, b, axis=2).pvalue
print(f"simulated, 20 null tests: {(p<0.05).any(axis=1).mean():.1%} of runs find 'something'")
print(f"average number of false positives per run: {(p<0.05).sum(axis=1).mean():.2f}")

This is not a subtle statistical issue. It is arithmetic.

**Where it bites in practice:** testing ten metrics in an A/B test, comparing every pair of
groups, or scanning a wide dataframe for interesting correlations. A 20-column correlation
matrix is **190** tests.

**The fixes:** Bonferroni (use 0.05/k — simple and strict), Benjamini–Hochberg (controls the
false discovery rate — the usual choice for many tests), or best of all, decide which test
matters before you look.

## 5.2 ⚠️ p-hacking: two habits that feel completely reasonable

In [ ]:
rng = np.random.default_rng(283)
T = 4000

#  (a) one honest test at n = 100
x = rng.normal(0, 1, (T, 100)); y = rng.normal(0, 1, (T, 100))
honest = (stats.ttest_ind(x, y, axis=1).pvalue < 0.05).mean()

#  (b) "let's check whether it's significant yet" - peek every 10 subjects
x = rng.normal(0, 1, (T, 100)); y = rng.normal(0, 1, (T, 100))
peeked = np.zeros(T, bool)
for m in range(20, 101, 10):
    peeked |= stats.ttest_ind(x[:, :m], y[:, :m], axis=1).pvalue < 0.05

#  (c) "it didn't work overall - but does it work for new users?"
x = rng.normal(0, 1, (T, 200)); y = rng.normal(0, 1, (T, 200))
g = rng.integers(0, 4, (T, 200))
subs = stats.ttest_ind(x, y, axis=1).pvalue < 0.05
for grp in range(4):
    m = g == grp
    sub = np.array([stats.ttest_ind(x[i][m[i]], y[i][m[i]]).pvalue for i in range(T)])
    subs |= sub < 0.05

print("False positive rate, on data with NO effect at all:")
print(f"  one honest test at n=100          {honest:.1%}")
print(f"  peeking every 10 from n=20 to 100 {peeked.mean():.1%}")
print(f"  overall test plus 4 subgroups     {subs.mean():.1%}")

> *"Let's check whether it's significant yet before spending more budget."*
>
> *"It didn't work overall — but does it work for new users?"*

Both are sensible-sounding sentences. Both triple or quadruple your false positive rate.

**Why:** every extra look is an extra chance for noise to cross the line. You are running many
tests — you are just not counting them.

And this is not only a problem on null data. Here is the *real* effect from section 1, with the
p-value computed as the data comes in:

In [ ]:
rng = np.random.default_rng(2801)
before = rng.normal(72, 11, 60)
after  = before + rng.normal(3.2, 6, 60)
change = after - before          # the effect is genuinely +3.2

print("A REAL effect, watched as the data arrives:")
for k in [10, 15, 20, 25, 30, 40, 60]:
    res = stats.ttest_1samp(change[:k], 0)
    mark = "  <- would have stopped here" if res.pvalue < 0.05 and k < 30 else ""
    print(f"  after {k:>2} people: mean {change[:k].mean():+.2f}  "
          f"p = {res.pvalue:.4f}  {'SIGNIFICANT' if res.pvalue<0.05 else 'not significant'}{mark}")

The effect is real throughout. The p-value still crosses the threshold at n = 15, goes back
above it at n = 25, and only settles later. A researcher peeking would have stopped at 15 and
reported a much larger effect than the real one.

**The fix:** write down the analysis before you see the data, and count every test you run.

## 5.3 ⚠️ The base rate — p < 0.05 does not mean 95% sure

In [ ]:
def ppv(prior, power, alpha=0.05):
    """Of the results that come back significant, what fraction are real?"""
    return prior*power / (prior*power + (1-prior)*alpha)

print(f"{'% of your hypotheses that are true':>36} {'80% power':>11} {'30% power':>11}")
for prior in [0.50, 0.20, 0.10, 0.05, 0.01]:
    print(f"{prior:>35.0%} {ppv(prior,0.80):>11.1%} {ppv(prior,0.30):>11.1%}")
print()
print("Worked through, with 1,000 hypotheses of which 100 are true, 80% power:")
print("   80 true findings")
print("   45 false alarms  (5% of the 900 that were not true)")
print("   so 45 of 125 'discoveries' - more than a third - are wrong")

If only 5% of what you test is real — which is optimistic for exploratory work — then **fewer
than half** your significant results are.

The p-value cannot see this, because it never asked how plausible your hypothesis was. This is
arithmetic, and it explains the replication crisis better than any accusation of misconduct
does.

**What moves the number:** testing plausible hypotheses rather than fishing, higher power
(bigger studies), and a stricter threshold where a false alarm is costly. None of these is
about the p-value itself.

## 5.4 ⚠️ Significant means detectable, not important

In [ ]:
#  a genuinely trivial difference: d = 0.02
d_true = 0.02
print(f"A difference of d = {d_true} - utterly trivial - at different sample sizes:")
print(f"{'n/group':>9} {'p-value':>12} {'Cohen d':>10}")
for n in [30, 100, 1000, 10_000, 100_000]:
    p = 2*(1 - stats.norm.cdf(d_true*np.sqrt(n/2)))
    print(f"{n:>9} {p:>12.2e} {d_true:>10.2f}")
print()
print("The p-value collapses. The effect size does not move at all.")
print("No real difference is exactly zero, so with enough data")
print("EVERYTHING becomes 'significant'.")

---
# 6. What to report instead of a verdict

**❌ The usual report**

> *"The treatment group performed significantly better (p < 0.05)."*

A reader cannot tell how big the effect was, how precisely it was measured, how many tests you
ran, or whether the difference matters.

**✅ A report worth reading**

> *"The course raised scores by 2.6 points (95% CI 1.0 to 4.1; d = 0.43; paired t-test, n = 60,
> p = 0.001). This was the single pre-specified comparison."*

**Five things to include, every time:** the effect size in real units · a confidence interval ·
the test and n · the exact p-value, not "< 0.05" · and how many tests you ran.

The last one is the only one that requires any courage, and it is the one that makes the other
four trustworthy.

## Mini build — a test that reports honestly

In [ ]:
def compare(a, b, n_tests=1, paired=False, label=""):
    """A two-group comparison that reports what matters.

    n_tests is deliberately a required thought: you have to say how many
    comparisons you ran, because that is what decides the real threshold.
    """
    a, b = np.asarray(a, float), np.asarray(b, float)

    if paired:
        d   = b - a
        res = stats.ttest_rel(b, a)
        est, df = d.mean(), len(d) - 1
        se  = d.std(ddof=1) / np.sqrt(len(d))
        cohen = d.mean() / d.std(ddof=1)
    else:
        res = stats.ttest_ind(a, b, equal_var=False)        # Welch, not the default
        est = b.mean() - a.mean()
        se  = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
        df  = res.df
        cohen = est / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)

    lo, hi = stats.t.interval(0.95, df, est, se)
    alpha  = 0.05 / n_tests                                  # Bonferroni

    print(f"{label}difference {est:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")
    print(f"   d = {cohen:+.3f}   p = {res.pvalue:.4f}   "
          f"threshold {alpha:.4f} ({n_tests} test(s))")
    print("   ->", "significant" if res.pvalue < alpha else "not significant")
    return est, (lo, hi), cohen, res.pvalue

In [ ]:
rng = np.random.default_rng(2801)
before = rng.normal(72, 11, 60)
after  = before + rng.normal(3.2, 6, 60)

print("the training course, reported honestly:")
compare(before, after, n_tests=1, paired=True, label="   ")

print("\nand the same result, if it had been one of ten metrics we checked:")
compare(before, after, n_tests=10, paired=True, label="   ")

#  a weaker effect, where admitting the ten tests actually flips the verdict
weak = before + np.random.default_rng(101).normal(1.6, 6, 60)
print("\na weaker effect, reported as the only comparison:")
compare(before, weak, n_tests=1, paired=True, label="   ")
print("\nthe same weaker effect, once we admit we checked ten metrics:")
compare(before, weak, n_tests=10, paired=True, label="   ")

Notice what happens with the weaker effect: identical data, identical estimate, identical
confidence interval — and the verdict flips purely because we told the truth about how many
metrics we looked at. Nothing about the evidence changed; only the honesty did.

Making `n_tests` a required argument forces you to answer the question nobody asks themselves.

It cannot stop you peeking, though. Only your study design can do that.

---

# Debug corner — eight testing mistakes

| You did | What goes wrong | Fix |
|---|---|---|
| read p as "chance the null is true" | it assumes the null; it cannot judge it | p = P(data \| null) |
| said "p > 0.05, so no effect" | you may just have had a small study | report the confidence interval |
| used `ttest_ind` with the default | `equal_var=True` fires 44% on some data | `equal_var=False` |
| used `ttest_ind` on before/after | between-person noise swamps it | `ttest_rel` — the data is paired |
| tested 20 metrics, reported the best | 64% chance of a false positive | Bonferroni or Benjamini–Hochberg |
| checked significance as data arrived | 5% becomes 16% | fix n in advance |
| reported p but not the effect size | a trivial difference looks like a finding | lead with the effect and its CI |
| said "p < 0.05" | 0.049 and 0.004 are different | report the exact value |

Rows 3 and 4 are one-word fixes that change the answer completely. Rows 5 and 6 are the ones
that get published.

In [ ]:
#  proof that rows 3 and 4 really do change the verdict
rng = np.random.default_rng(999)

#  row 3: unequal variance and unequal n
a = rng.normal(0, 5, 10); b = rng.normal(0, 1, 50)      # no real difference
print("row 3 - the same two groups, two tests:")
print(f"  Student (default): p = {stats.ttest_ind(a,b).pvalue:.4f}")
print(f"  Welch            : p = {stats.ttest_ind(a,b,equal_var=False).pvalue:.4f}")

#  row 4: paired data
person = rng.normal(100, 20, 20)
bef = person + rng.normal(0, 3, 20)
aft = person + rng.normal(0, 3, 20) + 4
print("\nrow 4 - the same before/after data, two tests:")
print(f"  unpaired: p = {stats.ttest_ind(bef, aft).pvalue:.4f}")
print(f"  paired  : p = {stats.ttest_rel(aft, bef).pvalue:.6f}")

---
# Recap — today in twelve lines

1. A test asks: is this gap big compared with yesterday's standard error?
2. `t = (what we saw − what we expected) / SE`. That is the whole formula.
3. A p-value is **P(data this extreme | nothing is happening)**. Nothing more.
4. Under the null, p is **uniform** — which is why 5% of nothing looks like something.
5. ⚠️ p is **not** the chance the null is true, and **not** the chance you are right.
6. ⚠️ p = 0.049 vs 0.051 is a gap of **0.017** in the test statistic.
7. ⚠️ Use Welch (`equal_var=False`). scipy's default fires 44% on some data.
8. ⚠️ Paired data needs `ttest_rel` — it took one example from 0% to 98%.
9. Power is the chance of seeing a real effect. 80% needs n = 25, 63 or 393 by effect size.
10. ⚠️ "Not significant" ≠ "no effect". Report the interval, which tells them apart.
11. ⚠️ 20 tests → 64% false positive. Peeking → 16%. Subgroups → 20%.
12. ⚠️ If 5% of your hypotheses are true, under half your "findings" are.

### Homework
- Find a published claim with a p-value and work out what is missing from the report.
- Take any two groups in your Day 21 project and compare them properly, reporting all five
  things.
- Work out what power your comparison had.

### Next class — Topic 2.9: Correlation
Measuring whether two variables move together — and the famous warning that goes with it,
which turns out to be more interesting than the slogan suggests.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*